# Hopf Bifurcation Analysis with KoopSim

## What is a Hopf bifurcation?

A **Hopf bifurcation** occurs when a fixed point of a dynamical system loses stability and gives rise to a periodic orbit (limit cycle). The normal form is:

$$\dot{x} = \mu x - y - x(x^2 + y^2)$$
$$\dot{y} = x + \mu y - y(x^2 + y^2)$$

For $\mu > 0$, the origin is unstable and trajectories spiral outward to a stable limit cycle of radius $\sqrt{\mu}$.

## Why Koopman?

The Hopf system is **nonlinear**, but the Koopman operator framework lifts it into a higher-dimensional space where the dynamics become (approximately) linear. This means we can:
- **Train once** on trajectory data
- **Predict at any future time** via matrix exponential -- no time-stepping required
- **Analyze the spectrum** to extract oscillation frequency and growth/decay rates

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from koopsim import KoopSim
from koopsim.systems import HopfBifurcation
from koopsim.utils.visualization import (
    plot_eigenspectrum,
    plot_phase_portrait,
    plot_trajectory_comparison,
)

## 1. Generate Training Data

We create a Hopf bifurcation system with $\mu = 1.0$ (stable limit cycle of radius 1) and generate snapshot pairs from multiple trajectories.

In [ ]:
system = HopfBifurcation(mu=1.0)
print(f"System: {system.name}, state dimension: {system.state_dim}")

# Generate snapshot pairs from multiple trajectories
x0 = np.array([0.5, 0.0])
dt = 0.01
X_train, Y_train = system.generate_snapshots(x0, dt=dt, n_steps=200, n_trajectories=15)
print(f"Training data: {X_train.shape[0]} snapshot pairs, {X_train.shape[1]} dimensions")

## 2. Train EDMD with Polynomial Dictionary

We use Extended Dynamic Mode Decomposition with a degree-3 polynomial dictionary. The polynomial basis captures the cubic nonlinearity in the Hopf normal form.

In [ ]:
sim = KoopSim(method="edmd", poly_degree=3)
sim.fit(X_train, Y_train, dt=dt)
print(sim)

## 3. Predict Trajectory and Compare to Ground Truth

The key Koopman advantage: we predict at arbitrary future times via `expm(L * t)` -- **no time-stepping loop**.

In [ ]:
# Generate ground truth trajectory
x0_test = np.array([0.3, 0.1])
traj_true = system.generate_trajectory(x0_test, dt=dt, n_steps=500)
times = np.arange(501) * dt

# Koopman prediction at the same times
traj_pred = sim.predict_trajectory(x0_test, times)

print(f"Ground truth shape: {traj_true.shape}")
print(f"Predicted shape:    {traj_pred.shape}")

## 4. Validate with RMSE

In [ ]:
# One-step prediction error on held-out data
x0_val = np.array([0.7, -0.2])
X_val, Y_val = system.generate_snapshots(x0_val, dt=dt, n_steps=100, n_trajectories=5)

rmse = sim.validate(X_val, Y_val, metric="rmse")
rel_err = sim.validate(X_val, Y_val, metric="relative")
print(f"One-step RMSE:      {rmse:.6e}")
print(f"Relative error:     {rel_err:.6e}")

# Multi-step trajectory RMSE
error_per_step = np.sqrt(np.mean((traj_true - traj_pred) ** 2, axis=1))
print(f"Mean trajectory RMSE: {np.mean(error_per_step):.6e}")
print(f"Max trajectory RMSE:  {np.max(error_per_step):.6e}")

## 5. Visualize: Phase Portrait

In [ ]:
fig = plot_phase_portrait([traj_true, traj_pred])
# Customize the legend
ax = fig.axes[0]
ax.legend(["Ground truth", "", "Koopman prediction", ""])
ax.set_title("Hopf Bifurcation -- Phase Portrait")
plt.show()

## 6. Visualize: Trajectory Comparison

In [ ]:
fig = plot_trajectory_comparison(times, traj_true, traj_pred, labels=["x", "y"])
plt.show()

## 7. Visualize: Eigenspectrum

The Koopman eigenvalues reveal the system's intrinsic frequencies and stability. Eigenvalues inside the unit circle correspond to decaying modes; those on the circle correspond to persistent oscillations.

In [ ]:
fig = plot_eigenspectrum(sim.model)
plt.show()

In [ ]:
# Spectral analysis: extract frequencies and growth rates
spectral = sim.spectral_analysis()
print(f"Number of Koopman modes: {len(spectral['eigenvalues'])}")
print("\nDominant mode frequencies (rad/s):")
top_idx = spectral["dominant_mode_indices"][:5]
for i in top_idx:
    lam = spectral["eigenvalues"][i]
    freq = spectral["frequencies"][i]
    gr = spectral["growth_rates"][i]
    print(f"  Mode {i}: |lambda|={abs(lam):.4f}, freq={freq:.4f} rad/s, growth={gr:.4f}")

## 8. Uncertainty Quantification

KoopSim provides Monte Carlo UQ by perturbing the initial condition and computing ensemble statistics.

In [ ]:
uq_result = sim.predict_with_uncertainty(x0_test, t=3.0, n_samples=200, noise_scale=0.05)
print(f"Prediction mean at t=3.0: {uq_result['mean']}")
print(f"Prediction std at t=3.0:  {uq_result['std']}")
print(f"95th percentile:          {uq_result['percentiles'][95]}")

In [ ]:
# Visualize uncertainty across a time range
from koopsim.utils.visualization import plot_uncertainty_band

t_uq = np.linspace(0, 5, 50)
means, stds = [], []
for t_i in t_uq:
    res = sim.predict_with_uncertainty(x0_test, t=t_i, n_samples=100, noise_scale=0.05)
    means.append(res["mean"])
    stds.append(res["std"])

means = np.array(means)
stds = np.array(stds)
traj_true_uq = system.generate_trajectory(x0_test, dt=t_uq[1] - t_uq[0], n_steps=len(t_uq) - 1)

fig = plot_uncertainty_band(t_uq, means, stds, true=traj_true_uq)
plt.show()

## 9. Quick Start with `from_system` Helper

For convenience, `KoopSim.from_system()` generates data and fits in one call.

In [ ]:
sim_quick = KoopSim.from_system(
    HopfBifurcation(mu=1.0),
    dt=0.01,
    n_steps=200,
    n_trajectories=15,
    poly_degree=3,
)
# Predict instantly at t=10
state_at_10 = sim_quick.predict(x0_test, t=10.0)
print(f"State at t=10: {state_at_10}")

## 10. Note on Neural Koopman (Optional)

If PyTorch is installed, you can also train a neural Koopman autoencoder. This approach works better for high-dimensional systems where hand-crafted dictionaries are insufficient.

```python
# Requires: pip install koopsim[neural]
sim_neural = KoopSim(method="neural", latent_dim=8, max_epochs=50)
sim_neural.fit(X_train, Y_train, dt=dt)
traj_neural = sim_neural.predict_trajectory(x0_test, times)
```

For a 2D system like Hopf, EDMD with polynomial dictionary is typically sufficient and much faster to train.

## Summary

In this notebook we:
1. Generated training data from the Hopf bifurcation system
2. Trained an EDMD model with polynomial observables (degree 3)
3. Predicted trajectories at arbitrary future times -- no time-stepping
4. Validated with RMSE on held-out data
5. Visualized phase portraits, trajectory comparisons, and the Koopman eigenspectrum
6. Performed Monte Carlo uncertainty quantification

The Koopman approach captures the limit cycle dynamics and provides instant-time queries after a single training pass.